<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 그래프 학습 · 추천 시스템 · 05. VGAE: 확률적 그래프 잠재공간

## Variational Graph Auto-Encoders

- **원 논문:** [Variational Graph Auto-Encoders](https://arxiv.org/abs/1611.07308)
- **저자 / 발표:** Thomas N. Kipf, Max Welling · NeurIPS Workshop on Bayesian Deep Learning (2016)
- **난이도 / 예상 시간:** 중급 · 약 70분
- **선수 지식:** VAE, GCN, binary cross entropy
- **로컬 학습 데이터:** `data/field_curriculum/graph_recommendation.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** GCN posterior encoder, reparameterization, inner-product decoder로 링크를 복원합니다.

**축소하거나 생략한 부분:** Cora/Citeseer/PubMed link prediction 대신 18노드 전체 adjacency 복원을 수행합니다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | Eq. (1) | factorized Gaussian q(Z|X,A) | mu/logstd shape |
| Eq. (2) | inner-product Bernoulli decoder | 대칭 logit |
| Eq. (3) | ELBO=reconstruction−KL | loss 감소 |
| §2, GCN encoder 설명 | 공유 첫 층·분기 head | positive/negative link score |

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** A [N,N], X [N,D] → μ/logσ² [N,Z] → link score [N,N]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

random.seed(7)
np.random.seed(7)
torch.manual_seed(7)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(7)

DATA_PATH = Path("data/field_curriculum/graph_recommendation.npz")
assert DATA_PATH.exists(), f"먼저 로컬 학습 데이터를 준비하세요: {DATA_PATH}"
raw = np.load(DATA_PATH, allow_pickle=False)
adjacency = torch.tensor(raw["adjacency"], dtype=torch.float32)
features = torch.tensor(raw["features"], dtype=torch.float32)
labels = torch.tensor(raw["labels"], dtype=torch.long)
train_mask = torch.tensor(raw["train_mask"], dtype=torch.bool)
test_mask = torch.tensor(raw["test_mask"], dtype=torch.bool)
assert adjacency.ndim == 2 and adjacency.shape[0] == adjacency.shape[1] == len(features)
assert torch.allclose(adjacency, adjacency.T) and not torch.diagonal(adjacency).any()
edge_count = int(adjacency.sum().item() / 2)
print(f"nodes={len(features)}, edges={edge_count}, features={features.shape[1]}")
print(ACCELERATOR.summary())
print(
    "graph sampling/topology preprocessing stays on CPU, "
    "while neural training uses DEVICE"
)

## 포트폴리오 해설: 핵심 수식에서 실행 코드까지

$$\mathcal L=\mathbb E_{q(Z\mid X,A)}[\log p(A\mid Z)]-D_{KL}(q(Z\mid X,A)\Vert p(Z))$$

- **기호 정의:** Z는 잠재 노드 벡터, q는 GCN posterior, p(A|Z)는 inner-product decoder입니다.
- **수식의 역할:** 재구성 우도와 Gaussian KL을 함께 최적화해 확률적 그래프 표현을 얻습니다.
- **구현 이유:** 원문 전체를 축약하더라도 위 관계가 결과를 만드는 핵심이므로,
  이 수식을 전처리·`forward`·loss 경계로 나누어 직접 구현했습니다.
- **Task/코드 대응:** Task P의 `preprocess_graph`, 논문 전용 클래스의
  `forward`, `compute_loss`, `training_step`이 Eq. (1)을 구조화합니다.
- **입출력 shape:** 특징 [N, F], 인접행렬 [N, N] → μ/logσ [N, D] → edge logit [N, N]
- **평가:** ELBO 감소와 positive/negative edge score 분리
- **원문 대비 한계:** Cora/Citeseer/PubMed link prediction 대신 18노드 전체 adjacency 복원을 수행합니다.

코드를 읽을 때는 `Config → 전처리 → Module.forward → loss → train → evaluate`
순서로 추적하세요. 각 함수는 한 가지 책임만 갖도록 분리했습니다.

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** Eq. (1)
- **이 셀의 책임:** factorized Gaussian q(Z|X,A)
- **Tensor shape 계약:** A [N,N], X [N,D] → μ/logσ² [N,Z] → link score [N,N]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** mu/logstd shape. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def norm_adj(adj):
    a = adj + torch.eye(len(adj), device=adj.device, dtype=adj.dtype)
    d = a.sum(1).pow(-0.5)
    return d[:, None] * a * d[None, :]


a_hat = norm_adj(adjacency)
assert torch.allclose(a_hat, a_hat.T)

### 구현 단계 3 · 핵심 연산·모델

- **원문 위치:** Eq. (1)
- **이 셀의 책임:** factorized Gaussian q(Z|X,A)
- **Tensor shape 계약:** A [N,N], X [N,D] → μ/logσ² [N,Z] → link score [N,N]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** mu/logstd shape. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
class VGAEEncoder(nn.Module):
    def __init__(self, in_dim, hidden=12, latent=6):
        super().__init__()
        self.shared = nn.Linear(in_dim, hidden, bias=False)
        self.mu = nn.Linear(hidden, latent, bias=False)
        self.logstd = nn.Linear(hidden, latent, bias=False)

    def forward(self, x, a):
        h = F.relu(a @ self.shared(x))
        mu = a @ self.mu(h)
        logstd = (a @ self.logstd(h)).clamp(-4, 4)
        z = mu + torch.randn_like(mu) * logstd.exp()
        return z, mu, logstd


encoder = VGAEEncoder(features.shape[1])
z, mu, logstd = encoder(features, a_hat)
assert z.shape == mu.shape == logstd.shape == (len(features), 6)

### 구현 단계 4 · 평가·완료 증거

- **원문 위치:** Eq. (3)
- **이 셀의 책임:** ELBO=reconstruction−KL
- **Tensor shape 계약:** A [N,N], X [N,D] → μ/logσ² [N,Z] → link score [N,N]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** loss 감소. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
encoder = ACCELERATOR.move(encoder)
features_device, a_hat_device, target = ACCELERATOR.move(
    features, a_hat, adjacency.clone()
)
target.fill_diagonal_(0)
pos_weight = (target.numel() - target.sum()) / target.sum().clamp_min(1)
optimizer = torch.optim.Adam(encoder.parameters(), lr=0.035)
losses = []
for _ in range(180):
    z, mu, logstd = encoder(features_device, a_hat_device)
    logits = z @ z.T
    recon = F.binary_cross_entropy_with_logits(logits, target, pos_weight=pos_weight)
    kl = -0.5 * torch.mean(1 + 2 * logstd - mu.pow(2) - (2 * logstd).exp())
    loss = recon + 0.02 * kl
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(float(loss.detach().cpu()))
with torch.no_grad():
    _, mu, _ = encoder(features_device, a_hat_device)
    probability = torch.sigmoid(mu @ mu.T)
    eye = torch.eye(len(target), dtype=torch.bool, device=target.device)
    pos = target.bool()
    neg = ~pos & ~eye
    pos_score = float(probability[pos].mean())
    neg_score = float(probability[neg].mean())
assert losses[-1] < losses[0] and pos_score > neg_score
fig, ax = plt.subplots(1, 2, figsize=(8, 3))
ax[0].plot(losses)
ax[0].set_title("negative ELBO")
ax[1].imshow(probability.detach().cpu(), cmap="viridis", vmin=0, vmax=1)
ax[1].set_title("p(A|Z)")
plt.tight_layout()
plt.show()
print({"edge_score": round(pos_score, 3), "nonedge_score": round(neg_score, 3)})

### Task P1 · 전처리와 핵심 모델

먼저 graph 입력의 shape와 정규화 규칙을 고정하고, 논문의 핵심 연산을
`nn.Module`의 `forward`와 `compute_loss`로 분리합니다.

### 구현 단계 5 · 핵심 연산·모델

- **원문 위치:** Eq. (1) / Eq. (2) / Eq. (3) / §2, GCN encoder 설명
- **이 셀의 책임:** factorized Gaussian q(Z|X,A) / inner-product Bernoulli decoder / ELBO=reconstruction−KL / 공유 첫 층·분기 head
- **Tensor shape 계약:** A [N,N], X [N,D] → μ/logσ² [N,Z] → link score [N,N]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** mu/logstd shape / 대칭 logit / loss 감소 / positive/negative link score. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class PortfolioConfig:
    latent_dim: int = 6
    learning_rate: float = 0.03
    steps: int = 14


def preprocess_graph(graph: Tensor) -> Tensor:
    with_self = graph + torch.eye(len(graph), dtype=graph.dtype)
    degree = with_self.sum(1).clamp_min(1.0).pow(-0.5)
    return degree[:, None] * with_self * degree[None, :]


class VgaePortfolioModel(nn.Module):
    def __init__(self, config: PortfolioConfig):
        super().__init__()
        self.config = config
        self.shared = nn.Linear(features.shape[1], 12)
        self.mean = nn.Linear(12, config.latent_dim)
        self.log_variance = nn.Linear(12, config.latent_dim)

    def forward(self, inputs: Tensor, graph: Tensor) -> tuple[Tensor, Tensor, Tensor]:
        hidden = F.relu(graph @ self.shared(inputs))
        mean = graph @ self.mean(hidden)
        log_variance = (graph @ self.log_variance(hidden)).clamp(-5.0, 5.0)
        latent = mean + torch.randn_like(mean) * torch.exp(0.5 * log_variance)
        return latent @ latent.T, mean, log_variance

    def compute_loss(self, outputs: tuple[Tensor, Tensor, Tensor]) -> Tensor:
        edge_logits, mean, log_variance = outputs
        reconstruction = F.binary_cross_entropy_with_logits(
            edge_logits, adjacency_device
        )
        kl = -0.5 * (1 + log_variance - mean.square() - log_variance.exp()).mean()
        return reconstruction + 0.05 * kl

    def training_step(self, inputs: Tensor, graph: Tensor) -> Tensor:
        return self.compute_loss(self(inputs, graph))

### Task P2 · objective와 update

`fit_portfolio_model`은 gradient가 흐르는 parameter와 한 step의 순서를
드러냅니다. 논문의 목적함수가 코드의 어느 tensor에 적용되는지 확인하세요.

### 구현 단계 6 · 목적함수·학습 update

- **원문 위치:** Eq. (3)
- **이 셀의 책임:** ELBO=reconstruction−KL
- **Tensor shape 계약:** A [N,N], X [N,D] → μ/logσ² [N,Z] → link score [N,N]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** loss 감소. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def fit_portfolio_model(
    model: VgaePortfolioModel,
    inputs: Tensor,
    graph: Tensor,
) -> list[float]:
    optimizer = torch.optim.Adam(model.parameters(), lr=model.config.learning_rate)
    history = []
    for _ in range(model.config.steps):
        loss = model.training_step(inputs, graph)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        history.append(float(loss.detach().cpu()))
    return history

### Task P3 · 평가와 한계 기록

loss와 별개의 논문 지표를 계산하고, 로컬 합성 graph에서 얻은 수치가
원 논문의 대규모 benchmark 결과를 대신하지 않는다는 점을 기록합니다.

### 구현 단계 7 · 평가·완료 증거

- **원문 위치:** §2, GCN encoder 설명
- **이 셀의 책임:** 공유 첫 층·분기 head
- **Tensor shape 계약:** A [N,N], X [N,D] → μ/logσ² [N,Z] → link score [N,N]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** positive/negative link score. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def evaluate_portfolio_model(
    model: VgaePortfolioModel,
    inputs: Tensor,
    graph: Tensor,
) -> dict[str, float]:
    with torch.no_grad():
        logits, _, _ = model(inputs, graph)
        positive = logits[adjacency_device.bool()].mean()
        negative = logits[~adjacency_device.bool()].mean()
    return {"edge_score_gap": float((positive - negative).cpu())}


features_device, adjacency_device = ACCELERATOR.move(features, adjacency)
portfolio_config = PortfolioConfig()
portfolio_graph = ACCELERATOR.move(preprocess_graph(adjacency))
portfolio_model = ACCELERATOR.move(VgaePortfolioModel(portfolio_config))
portfolio_history = fit_portfolio_model(
    portfolio_model,
    features_device,
    portfolio_graph,
)
portfolio_metrics = evaluate_portfolio_model(
    portfolio_model,
    features_device,
    portfolio_graph,
)
assert np.isfinite(portfolio_history).all()
print(portfolio_metrics)

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> Cora/Citeseer/PubMed link prediction 대신 18노드 전체 adjacency 복원을 수행합니다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.